In [19]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing import TypedDict
import subprocess
from openai import OpenAI
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator
import base64
import textwrap

from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv()

# 환경 변수가 제대로 로드되었는지 확인 (디버깅용 - 나중에 제거 가능)
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️ OPENAI_API_KEY가 로드되지 않았습니다. .env 파일을 확인하세요.")
else:
    print("✅ OPENAI_API_KEY가 성공적으로 로드되었습니다.")

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file: str
    audio_file: str
    transcription : str
    summaries : Annotated[list[str], operator.add]
    final_summary: str
    thumbnail_prompt : Annotated[list[str], operator.add]
    thumbnail_sketches: Annotated[list[str], operator.add]
    final_thumbnail: str

✅ OPENAI_API_KEY가 성공적으로 로드되었습니다.


In [20]:
def extract_audio(state: State):
    import imageio_ffmpeg
    
    output_file = state["video_file"].replace("mp4", "mp3")
    # imageio-ffmpeg에서 제공하는 ffmpeg 실행 파일 경로 사용
    ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
    
    command = [
        ffmpeg_exe,
        "-y",  # 확인 없이 기존 파일 덮어쓰기
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        output_file
    ]
    subprocess.run(command)
    return {
        "audio_file" : output_file,
    }

In [21]:
def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="en",
            prompt="jennie", 
        )
        return {
            'transcription':transcription,
        }

In [22]:
import textwrap
from langgraph.types import Send

def dispatch_summarizers(state: State):
    transcription = state['transcription']
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id": i+1, "chunk" : chunk})
    return [Send("summarize_chunk", chunk) for chunk in chunks]

In [23]:

def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {
        "summaries" : [summary],
    }

In [24]:
from openai.resources.completions import CompletionsWithStreamingResponse


def mega_summary(state: State):
    all_summaries = "\n".join(state["summaries"])

    prompt = f"""
        You are given multiple summaries of different chunks from a vieo transcription.
        Please create a comprehecsive final summary that combines all the key points.
        Individual summaries:
        {all_summaries}
    """

    response = llm.invoke(prompt)

    return {
        "final_summary":response.content,
    }

def dispatch_artists(state:State):
    return [
        Send("generate_thumbnails",     
        {
            "id": i,
            "summary":state["final_summary"],
        })
        for i in [1,2,3,4,5]
    ]

def generate_thumbnails(args):
    concept_id = args["id"]
    summary = args["summary"]

    prompt = f"""
    Based on this video summary, create a detailed visual prompt for a YouTube thumbnail.

    Create a detailed prompt for generating a thumbnail image that would attract viewers. Inlcude:
        - Main visual elements
        - Color scheme
        - Text overlay suggestions
        - Overall composition
    
    Summary : {summary}
    """

    response = llm.invoke(prompt)

    thumbnail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model="gpt-image-1",
        prompt=thumbnail_prompt,
        quality="low",
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)

    filename = f"thumbnail_{concept_id}.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes)

    return {
        "thumbnail_prompts": [thumbnail_prompt],
        "thumbnail_sketches": [filename],
    }

In [25]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)
graph_builder.add_node("mega_summary", mega_summary)
graph_builder.add_node("generate_thumbnails", generate_thumbnails)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges(
    "transcribe_audio", dispatch_summarizers, ["summarize_chunk"]
)
graph_builder.add_edge("summarize_chunk", "mega_summary")
graph_builder.add_conditional_edges(
    "mega_summary", dispatch_artists, ["generate_thumbnails"]
)
graph_builder.add_edge("generate_thumbnails", END)

graph = graph_builder.compile()

In [26]:
graph.invoke({"video_file": "video.mp4"})

{'video_file': 'video.mp4',
 'audio_file': 'video.mp3',
 'transcription': 'if we had the nails on my jennie i was gonna make since like like like like jennie jennie jennie i think i really like jennie jennie haters gonna really like jennie jennie\n',
 'summaries': ['[Chunk 1] The text expresses a strong fondness for someone named Jennie, repeating her name and emphasizing the feelings of admiration despite potential criticism from others.'],
 'final_summary': 'The text conveys a deep admiration for someone named Jennie, highlighting strong feelings of affection and appreciation. It acknowledges that this fondness may invite criticism from others, but the admiration remains steadfast and unwavering.',
 'thumbnail_prompt': [],
 'thumbnail_sketches': ['thumbnail_1.jpg',
  'thumbnail_2.jpg',
  'thumbnail_3.jpg',
  'thumbnail_4.jpg',
  'thumbnail_5.jpg']}